<a href="https://colab.research.google.com/github/JavierRamirez14/CNN_TFG/blob/master/ejecucion_gpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TFG | FUNCIONAMIENTO Y APLICACIÓN DE REDES NEURONALES PARA LA CLASIFICACIÓN DE IMÁGENES

#### *Javier Ramírez Abad*

## 0. Importaciones

In [1]:
import os
import shutil
import numpy as np
import cupy as cp
from PIL import Image
import time
from google.colab import files
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc, precision_recall_curve, average_precision_score
import pandas as pd
from itertools import cycle
import pickle
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, MaxPooling2D, Flatten, Dropout
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.applications import ResNet50
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc, precision_recall_curve, average_precision_score

## 1. Desarrollo de una CNN desde cero

In [10]:
# capa.py

# Estructura general de una capa
class Capa:
    def __init__(self):
        self.input = None # Almacena la entrada de la capa
        self.output = None # Almacena la salida de la capa

    def forward(self, input):
        # Propagación hacia adelante
        pass

    def backward(self, output_gradient, learning_rate):
        # Retropropagación
        pass


# dense.py

class Dense(Capa):
    """
    Capa densa (fully connected).

    Args:
        n_input (int): Número de neuronas en la capa de entrada.
        n_output (int): Número de neuronas en la capa de salida.
    """
    # Inicialización Xavier/Glorot para los pesos
    def __init__(self, n_input, n_output):
        limit = cp.sqrt(6 / (n_input + n_output))
        self.weights = cp.random.uniform(-limit, limit, (n_input, n_output))
        self.bias = cp.zeros((1, n_output)) # Inicialización de bias a cero

    def forward(self, input):
        """Calcula la salida de la capa densa: input * weights + bias."""
        self.input = input
        m = cp.dot(self.input, self.weights)
        return m + self.bias

    def backward(self, input_gradient, learning_rate):
        """Propaga el gradiente hacia atrás y actualiza pesos/bias."""
        # Gradiente de los pesos (transpuesta de input * gradiente de entrada)
        weights_gradient = cp.dot(self.input.T, input_gradient)
        # Gradiente para la capa anterior (gradiente de entrada * transpuesta de weights)
        output_gradient = cp.dot(input_gradient, self.weights.T)
        # Gradiente del bias (suma sobre el batch)
        bias_gradient = cp.sum(input_gradient, axis=0, keepdims=True)

        # Actualización de parámetros
        self.weights -= learning_rate * weights_gradient
        self.bias -= learning_rate * bias_gradient

        return output_gradient


#activations.py

class ReLU(Capa):
    def forward(self, input):
        """Aplica ReLU: max(0, input)."""
        self.input = input
        return cp.maximum(0, self.input)

    def backward(self, output_gradient, learning_rate):
        """Gradiente de ReLU: 1 si input > 0, 0 en otro caso."""
        return (self.input > 0) * output_gradient

class Sigmoid(Capa):
    def forward(self, input):
        """Aplica sigmoide: 1/(1 + e^-input)."""
        self.input = input
        return 1 / (1 + cp.exp(-self.input))

    def backward(self, input_gradient, learning_rate):
        """Gradiente de sigmoide: sigmoid(x)*(1-sigmoid(x))."""
        s = self.forward(self.input)
        return s * (1 - s) * input_gradient

class Softmax(Capa):
    def forward(self, input):
        """Aplica softmax con estabilidad numérica."""
        self.input = input
        exp_input = cp.exp(input - cp.max(input, axis=1, keepdims=True))
        self.output = exp_input / cp.sum(exp_input, axis=1, keepdims=True)
        return self.output

    def backward(self, input_gradient, learning_rate):
        """Gradiente de softmax."""
        batch_size = self.output.shape[0]
        grad = cp.zeros_like(self.output)

        for i in range(batch_size):
            single_output = self.output[i].reshape(-1, 1)
            jacobian = cp.diagflat(single_output) - cp.dot(single_output, single_output.T)
            grad[i] = cp.dot(jacobian, input_gradient[i])

        return grad


#loss.py

"""
    Todas las funciones de pérdida reciben:
        y_true (array): Valores reales.
        y_pred (array): Predicciones del modelo.
"""

def mse(y_true, y_pred):
    """Calcula el error cuadrático medio."""
    return cp.mean((y_true - y_pred)**2)

def mse_prime(y_true, y_pred):
    """Derivada del MSE respecto a y_pred."""
    return 2 * (y_pred - y_true) / cp.size(y_true)

def accuracy(y_true, y_pred):
    """Calcula la precisión (porcentaje de aciertos)."""
    y_pred = np.argmax(y_pred, axis=1)
    y_true = np.argmax(y_true, axis=1)
    correct = cp.sum(y_true == y_pred)
    total = y_true.shape[0]
    return correct / total

def categorical_cross_entropy(y_true, y_pred):
    """Calcula la pérdida de entropía cruzada categórica."""
    # Evitar log(0) agregando un pequeño valor epsilon
    epsilon = 1e-12
    y_pred = cp.clip(y_pred, epsilon, 1.0 - epsilon)

    # Calcular la pérdida
    loss = -cp.sum(y_true * cp.log(y_pred)) / y_pred.shape[0]  # Promedio sobre el batch
    return loss


def categorical_cross_entropy_prime(y_true, y_pred):
    """Calcula el gradiente de la pérdida de entropía cruzada categórica con respecto a y_pred."""
    # El gradiente es (y_pred - y_true) / batch_size
    batch_size = y_pred.shape[0]
    return (y_pred - y_true) / batch_size


#convolution.py

class Convolution(Capa):
    """
    Versión optimizada de capa convolucional usando im2col.

    Args:
        input_shape (tuple): Forma de entrada (canales, altura, ancho).
        kernel_size (int): Tamaño del kernel (cuadrado).
        n_kernels (int): Número de filtros convolucionales.
    """
    def __init__(self, input_shape, kernel_size, n_kernels):
        depth, height, width = input_shape
        self.input_shape = input_shape
        self.depth = depth
        self.n_kernels = n_kernels
        self.kernel_size = kernel_size
        self.output_shape = (n_kernels, height - kernel_size + 1, width - kernel_size + 1)
        self.kernels_shape = (n_kernels, depth, kernel_size, kernel_size)

        # Xavier/Glorot initialization para los kernels
        limit = cp.sqrt(6 / (depth * kernel_size * kernel_size + n_kernels))
        self.kernels = cp.random.uniform(-limit, limit, self.kernels_shape)

        # Inicializar biases con la forma correcta
        self.biases = cp.zeros(self.output_shape)

    def _im2col(self, input_data, kernel_size):
        """Transforma la entrada en una matriz que permite operaciones vectorizadas para la convolución"""
        batch_size, depth, height, width = input_data.shape
        out_height = height - kernel_size + 1
        out_width = width - kernel_size + 1

        # Inicializar la matriz de salida
        col = cp.zeros((batch_size, depth, kernel_size, kernel_size, out_height, out_width))

        # Llenar la matriz con los valores correspondientes
        for y in range(kernel_size):
            y_max = y + out_height
            for x in range(kernel_size):
                x_max = x + out_width
                col[:, :, y, x, :, :] = input_data[:, :, y:y_max, x:x_max]

        # Reshape para obtener la matriz final
        col = col.transpose(0, 4, 5, 1, 2, 3).reshape(batch_size * out_height * out_width, -1)
        return col

    def _col2im(self, col, input_shape):
        """Función inversa de im2col para la propagación hacia atrás"""
        batch_size, depth, height, width = input_shape
        kernel_size = self.kernel_size
        out_height = height - kernel_size + 1
        out_width = width - kernel_size + 1

        # Inicializar la matriz de salida
        img = cp.zeros((batch_size, depth, height, width))

        # Reshapear col para facilitar la operación inversa
        col_reshaped = col.reshape(batch_size, out_height, out_width, depth, kernel_size, kernel_size)
        col_reshaped = col_reshaped.transpose(0, 3, 4, 5, 1, 2)

        # Sumar los valores a la matriz de salida
        for y in range(kernel_size):
            y_max = y + out_height
            for x in range(kernel_size):
                x_max = x + out_width
                img[:, :, y:y_max, x:x_max] += col_reshaped[:, :, y, x, :, :]

        return img

    def forward(self, input):
        """Realiza la convolución utilizando operaciones matriciales"""
        self.input = input
        batch_size, depth, height, width = input.shape

        # Crear la matriz de columnas
        self.col = self._im2col(input, self.kernel_size)

        # Reshape kernels para multiplicación matricial
        kernels_reshaped = self.kernels.reshape(self.n_kernels, -1)

        # Realizar la convolución como una multiplicación de matrices
        output = cp.dot(self.col, kernels_reshaped.T)

        # Reshape para obtener el resultado final
        output = output.reshape(batch_size, height - self.kernel_size + 1,
                                width - self.kernel_size + 1, self.n_kernels)
        output = output.transpose(0, 3, 1, 2)

        # Añadir biases (asegurándose de que la forma sea correcta para broadcasting)
        output += self.biases[cp.newaxis, :, :, :]

        return output

    def backward(self, input_gradient, learning_rate):
        """Propaga el gradiente hacia atrás"""
        batch_size = input_gradient.shape[0]

        # Reshape input_gradient para multiplicación matricial
        input_gradient_reshaped = input_gradient.transpose(0, 2, 3, 1).reshape(-1, self.n_kernels)

        # Calcular el gradiente de los kernels
        kernels_gradient = cp.dot(input_gradient_reshaped.T, self.col)
        kernels_gradient = kernels_gradient.reshape(self.kernels_shape)

        # Calcular el gradiente de la entrada
        col_gradient = cp.dot(input_gradient_reshaped, self.kernels.reshape(self.n_kernels, -1))
        output_gradient = self._col2im(col_gradient, self.input.shape)

        # Calcular el gradiente del bias y asegurarse de que tenga la forma correcta
        # Sumar sobre el batch (eje 0) y mantener las dimensiones espaciales
        biases_gradient = cp.sum(input_gradient, axis=0)

        # Actualizar parámetros
        self.kernels -= learning_rate * kernels_gradient
        self.biases -= learning_rate * biases_gradient

        return output_gradient


#pooling.py

class Pooling(Capa):
    """
    Versión optimizada de la capa de max pooling.

    Args:
        kernel_size (int): Tamaño de la ventana de pooling (cuadrada).
        stride (int): Paso de desplazamiento de la ventana (usualmente igual a kernel_size).
    """
    def __init__(self, kernel_size, stride):
        self.kernel_size = kernel_size
        self.stride = stride
        self.input_shape = None
        self.output_shape = None
        self.h_indices = None  # Array para almacenar índices verticales
        self.w_indices = None  # Array para almacenar índices horizontales

    def forward(self, input):
        """Aplica la operación de max pooling sobre la entrada."""
        self.input = input
        batch_size, depth, height, width = input.shape
        self.input_shape = input.shape

        # Calcular las dimensiones de la salida
        out_height = (height - self.kernel_size) // self.stride + 1
        out_width = (width - self.kernel_size) // self.stride + 1
        self.output_shape = (batch_size, depth, out_height, out_width)

        # Crear una vista de la entrada para aplicar el pooling
        input_reshaped = input.reshape(batch_size * depth, 1, height, width)
        input_strided = cp.lib.stride_tricks.as_strided(
            input_reshaped,
            shape=(batch_size * depth, out_height, out_width, self.kernel_size, self.kernel_size),
            strides=(input_reshaped.strides[0],
                    input_reshaped.strides[2] * self.stride,
                    input_reshaped.strides[3] * self.stride,
                    input_reshaped.strides[2],
                    input_reshaped.strides[3])
        )

        # Aplicar max pooling
        output = cp.max(input_strided, axis=(3, 4)).reshape(batch_size, depth, out_height, out_width)

        # Obtener los índices de los máximos
        max_indices = cp.argmax(input_strided.reshape(batch_size * depth, out_height, out_width, -1), axis=3)
        self.h_indices = (max_indices // self.kernel_size).reshape(batch_size, depth, out_height, out_width)
        self.w_indices = (max_indices % self.kernel_size).reshape(batch_size, depth, out_height, out_width)

        return output

    def backward(self, input_gradient, learning_rate):
        """Realiza la retropropagación para la capa de pooling."""
        batch_size, depth, out_height, out_width = input_gradient.shape
        output_gradient = cp.zeros(self.input_shape)

        # Calcular las coordenadas en la imagen original
        h_coords = self.h_indices + cp.arange(out_height).reshape(1, 1, -1, 1) * self.stride
        w_coords = self.w_indices + cp.arange(out_width).reshape(1, 1, 1, -1) * self.stride

        # Asignar el gradiente a las posiciones de los máximos
        for img in range(batch_size):
            for d in range(depth):
                output_gradient[img, d, h_coords[img, d], w_coords[img, d]] += input_gradient[img, d]

        return output_gradient


# reshape.py

class Reshape(Capa):
    def __init__(self, input_shape, output_shape):
        """Inicializa capa de reshape."""
        self.input_shape = input_shape
        self.output_shape = output_shape

    def forward(self, input):
        """Cambia la forma del tensor de entrada."""
        self.batch_size = input.shape[0]
        return cp.reshape(input, (self.batch_size, self.output_shape))

    def backward(self, input_gradient, learning_rate):
        """Revierte el reshape a la forma original."""
        return cp.reshape(input_gradient, (self.batch_size, *self.input_shape))


# network.py

def train_batch(X, y, net, loss, loss_prime, learning_rate):
    """
    Entrena un batch completo de datos y devuelve el error promedio.

    Args:
        X (numpy.ndarray): Datos de entrada.
        y (numpy.ndarray): Etiquetas verdaderas.
        net (list): Lista de capas de la red.
        loss (function): Función de pérdida.
        loss_prime (function): Derivada de la función de pérdida.
        learning_rate (float): Tasa de aprendizaje.
    """
    # Forward
    input = X
    for layer in net:
        input = layer.forward(input)
    y_pred = input

    # Error
    error = loss(y, y_pred)

    # Backward
    grad = loss_prime(y, y_pred)
    for layer in reversed(net):
        grad = layer.backward(grad, learning_rate)

    # Devolver error, predicciones y etiquetas verdaderas
    return error, y_pred, y

def train(train_data, val_data, net, loss, loss_prime, epochs, learning_rate):
    """
    Entrena la red durante varias épocas.

    Args:
        data (iterable): Iterable que proporciona batches de datos (X, y).
        net (list): Lista de capas que forman la red neuronal.
        loss (function): Función de pérdida a utilizar (ej: mse).
        loss_prime (function): Derivada de la función de pérdida.
        epochs (int): Número de épocas de entrenamiento.
        learning_rate (float): Tasa de aprendizaje para la actualización de pesos.
    """
    history = {
        'loss': [],
        'accuracy': [],
        'val_loss': [],
        'val_accuracy': []
    }

    for epoch in range(epochs):
        start_time = time.time()
        epoch_error = 0
        batch_count = 0

        # Almacenar predicciones y etiquetas verdaderas de entrenamiento
        for X, y in train_data:
            y_train_preds = cp.empty((0, y.shape[1]))
            y_train_true = cp.empty((0, y.shape[1]))
            break

        for X, y in train_data:
            # Convertir inputs a CuPy si no lo están ya
            X = cp.asarray(X)
            y = cp.asarray(y)

            # Usar la función train_batch
            error, batch_preds, batch_true = train_batch(X, y, net, loss, loss_prime, learning_rate)

            # Asegurarse de que los batch outputs son CuPy arrays
            batch_preds = cp.asarray(batch_preds)
            batch_true = cp.asarray(batch_true)

            # Acumular predicciones y etiquetas
            y_train_preds = cp.concatenate((y_train_preds, batch_preds), axis=0)
            y_train_true = cp.concatenate((y_train_true, batch_true), axis=0)

            batch_count += 1
            epoch_error += error

        # Calcular métricas de entrenamiento
        epoch_error = float(epoch_error / batch_count)
        train_acc = float(accuracy(y_train_true, y_train_preds))

        # Calcular métricas en validación
        val_error, val_acc = evaluate(val_data, net, loss)
        val_error = float(val_error)
        val_acc = float(val_acc)

        # Mostrar métricas
        print(f'Epoch: {epoch+1}/{epochs} | Train Loss: {epoch_error:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_error:.4f} | Val Acc: {val_acc:.4f}')

        # Guardar métricas
        history['loss'].append(epoch_error)
        history['accuracy'].append(train_acc)
        history['val_loss'].append(val_error)
        history['val_accuracy'].append(val_acc)

        print(f"Tiempo de época: {time.time() - start_time:.2f} segundos")
        cp.cuda.Device().synchronize()

    return history

def test(X, y, net, batch_size=64):
    """Evalúa la red y devuelve el accuracy."""
    # Número de muestras
    num_samples = X.shape[0]

    # Listas para almacenar las predicciones y las etiquetas verdaderas
    y_pred_list = []
    y_true_list = []

    # Procesar los datos en mini-batches
    for i in range(0, num_samples, batch_size):
        # Obtener el mini-batch actual
        X_batch = X[i:i + batch_size]
        y_batch = y[i:i + batch_size]

        # Forward pass
        input = X_batch
        for layer in net:
            input = layer.forward(input)

        # Almacenar las predicciones y las etiquetas verdaderas
        y_pred_list.append(input)
        y_true_list.append(y_batch)

    # Concatenar todos los mini-batches
    y_pred = cp.concatenate(y_pred_list)
    y_true = cp.concatenate(y_true_list)

    # Calcular el accuracy
    return y_true, y_pred


def evaluate(data, net, loss):
    """Evalúa la red en los datos de validación."""
    for X, y in data:
            y_preds = cp.empty((0, y.shape[1]))
            y_true = cp.empty((0, y.shape[1]))
            break

    for X, y in data:
        # Convertir inputs a CuPy
        X = cp.asarray(X)
        y = cp.asarray(y)

        # Forward pass
        output = X
        for layer in net:
            output = layer.forward(output)

        # Asegurar que las predicciones son CuPy arrays
        y_preds = cp.concatenate((y_preds, cp.asarray(output)), axis=0)
        y_true = cp.concatenate((y_true, y), axis=0)

    # Calcular métricas
    error = float(loss(y_true, y_preds))
    acc = accuracy(y_true, y_preds)

    return error, acc

In [3]:
def guardar_modelo(modelo, ruta_archivo):
    """
    Guarda los parámetros del modelo en un archivo.

    Args:
        modelo: Lista de capas que conforman la red neuronal
        ruta_archivo: Ruta donde se guardará el archivo del modelo
    """
    # Crear un diccionario para almacenar los parámetros de cada capa
    parametros = []

    for i, capa in enumerate(modelo):
        parametros_capa = {}

        # Guardar parámetros según el tipo de capa
        if isinstance(capa, Dense):
            parametros_capa['tipo'] = 'Dense'
            parametros_capa['weights'] = cp.asnumpy(capa.weights)
            parametros_capa['bias'] = cp.asnumpy(capa.bias)

        elif isinstance(capa, Convolution):
            parametros_capa['tipo'] = 'Convolution'
            parametros_capa['kernels'] = cp.asnumpy(capa.kernels)
            parametros_capa['biases'] = cp.asnumpy(capa.biases)
            parametros_capa['input_shape'] = capa.input_shape
            parametros_capa['kernel_size'] = capa.kernel_size
            parametros_capa['n_kernels'] = capa.n_kernels

        elif isinstance(capa, Reshape):
            parametros_capa['tipo'] = 'Reshape'
            parametros_capa['input_shape'] = capa.input_shape
            parametros_capa['output_shape'] = capa.output_shape

        elif isinstance(capa, Pooling):
            parametros_capa['tipo'] = 'Pooling'
            parametros_capa['kernel_size'] = capa.kernel_size
            parametros_capa['stride'] = capa.stride

        elif isinstance(capa, ReLU):
            parametros_capa['tipo'] = 'ReLU'

        elif isinstance(capa, Sigmoid):
            parametros_capa['tipo'] = 'Sigmoid'

        elif isinstance(capa, Softmax):
            parametros_capa['tipo'] = 'Softmax'

        if parametros_capa:
            parametros.append(parametros_capa)

    # Guardar los parámetros en un archivo
    with open(ruta_archivo, 'wb') as f:
        pickle.dump(parametros, f)

    print(f"Modelo guardado en {ruta_archivo}")

def cargar_modelo(ruta_archivo):
    """
    Carga los parámetros del modelo desde un archivo.

    Args:
        ruta_archivo: Ruta del archivo donde está guardado el modelo

    Returns:
        Lista de capas que conforman la red neuronal con parámetros cargados
    """
    # Cargar los parámetros desde el archivo
    with open(ruta_archivo, 'rb') as f:
        parametros = pickle.load(f)

    # Crear el modelo con los parámetros cargados
    modelo = []

    for parametros_capa in parametros:
        tipo = parametros_capa['tipo']

        if tipo == 'Dense':
            n_input, n_output = parametros_capa['weights'].shape
            capa = Dense(n_input, n_output)
            capa.weights = cp.asarray(parametros_capa['weights'])
            capa.bias = cp.asarray(parametros_capa['bias'])

        elif tipo == 'Convolution':
            input_shape = parametros_capa['input_shape']
            kernel_size = parametros_capa['kernel_size']
            n_kernels = parametros_capa['n_kernels']
            capa = Convolution(input_shape, kernel_size, n_kernels)
            capa.kernels = cp.asarray(parametros_capa['kernels'])
            capa.biases = cp.asarray(parametros_capa['biases'])

        elif tipo == 'Reshape':
            capa = Reshape(parametros_capa['input_shape'], parametros_capa['output_shape'])

        elif tipo == 'Pooling':
            capa = Pooling(parametros_capa['kernel_size'], parametros_capa['stride'])

        elif tipo == 'ReLU':
            capa = ReLU()

        elif tipo == 'Sigmoid':
            capa = Sigmoid()

        elif tipo == 'Softmax':
            capa = Softmax()

        modelo.append(capa)

    print(f"Modelo cargado desde {ruta_archivo}")
    return modelo


## 2. Preparación del entorno virtual y descarga de datos

In [4]:
print("Verificando GPU disponible...")
try:
    print(f"Dispositivo CUDA: {cp.cuda.runtime.getDeviceCount()} dispositivo(s) disponible(s)")
    print(f"Usando dispositivo: {cp.cuda.runtime.getDeviceProperties(cp.cuda.Device().id)['name']}")
except Exception as e:
    print(f"Error al verificar GPU: {e}")
    print("Usando CPU en su lugar")

Verificando GPU disponible...
Dispositivo CUDA: 1 dispositivo(s) disponible(s)
Usando dispositivo: b'Tesla T4'


Puedes descargar el archivo "kaggle.json" desde tu cuenta de Kaggle accediendo a https://www.kaggle.com/settings y haciendo clic en el botón "Create New API Token", lo que generará automáticamente el archivo.

In [5]:
uploaded = files.upload()  # Esto te permitirá subir el archivo kaggle.json

if 'kaggle.json' in uploaded:
    print('kaggle.json subido correctamente.')
else:
    print('No se pudo subir el archivo kaggle.json.')

Saving kaggle.json to kaggle.json
kaggle.json subido correctamente.


In [6]:
%%capture
! pip install -q kaggle
! mkdir ~/.kaggle
! cp kaggle.json ~/.kaggle/
! chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset
! mkdir dataset
! unzip brain-tumor-mri-dataset.zip -d dataset

## 3. Preprocesamiento de datos

In [7]:
# Rutas de las carpetas
train_path = '/content/dataset/Training'
test_path = '/content/dataset/Testing'
clases = ['glioma', 'meningioma', 'notumor', 'pituitary']
n = len(clases)

# Tamaño fijo para redimensionar (ancho, alto)
size = (150, 150)

# Función para cargar imágenes desde una carpeta con formato (3, 64, 64)
def load_images_from_folder(folder_path):
    print(f"Cargando imágenes desde {folder_path}...")
    X = []
    y = []
    for i, clase in enumerate(clases):
        clase_path = os.path.join(folder_path, clase)
        count = 0
        for img_name in os.listdir(clase_path):
            img_path = os.path.join(clase_path, img_name)
            with Image.open(img_path) as img:
                # Redimensiona la imagen al tamaño fijo
                img = img.resize(size)
                # Convertir a RGB si no lo es
                if img.mode != "RGB":
                    img = img.convert("RGB")
                # Convertir la imagen a array y normalizar
                img_array = np.array(img).astype('float32') / 255.0
                # Reorganizar la dimensión para que sea (3, 64, 64) en lugar de (64, 64, 3)
                img_array = np.transpose(img_array, (2, 0, 1))
                # Agregar la imagen al dataset
                if img_array.shape == (3, 150, 150):
                    X.append(img_array)
                    # Crear la codificación one-hot
                    one_hot = [0] * n
                    one_hot[i] = 1
                    y.append(one_hot)
                    count += 1
        print(f"  Clase {clase}: {count} imágenes cargadas")
    return np.array(X), np.array(y)

# Función para crear mini-batches
def crear_mini_batches(X, y, batch_size, shuffle=True):
    if shuffle:
        indices = np.arange(X.shape[0])
        np.random.shuffle(indices)
        X = X[indices]
        y = y[indices]

    mini_batches = []
    for i in range(0, X.shape[0], batch_size):
        X_mini_batch = X[i:i + batch_size]
        y_mini_batch = y[i:i + batch_size]
        # Convertir a CuPy arrays
        X_mini_batch_gpu = cp.asarray(X_mini_batch)
        y_mini_batch_gpu = cp.asarray(y_mini_batch)
        mini_batches.append((X_mini_batch_gpu, y_mini_batch_gpu))

    return mini_batches

In [8]:
# Iniciar temporizador
start_time = time.time()

print("Iniciando carga de datos...")
# Cargar datos de entrenamiento
X_train_total, y_train_total = load_images_from_folder(train_path)
# Dividir los datos de entrenamiento en entrenamiento y validación (80% train, 20% validation)
X_train, X_val, y_train, y_val = train_test_split(X_train_total, y_train_total, test_size=0.2, random_state=42, stratify=y_train_total)
# Cargar datos de prueba
X_test, y_test = load_images_from_folder(test_path)

# Imprimir las formas de los datos
print("X_train shape:", X_train.shape)      # Debería ser (80% de num_imagenes, 3, 64, 64)
print("y_train shape:", y_train.shape)      # Debería ser (80% de num_imagenes, n)
print("X_val shape:", X_val.shape)          # Debería ser (20% de num_imagenes, 3, 64, 64)
print("y_val shape:", y_val.shape)          # Debería ser (20% de num_imagenes, n)
print("X_test shape:", X_test.shape)        # Debería ser (num_imagenes_test, 3, 64, 64)
print("y_test shape:", y_test.shape)        # Debería ser (num_imagenes_test, n)

# Crear mini-batches para entrenamiento
print("Creando mini-batches y transfiriendo a GPU...")
batch_size = 64
train_data = crear_mini_batches(X_train, y_train, batch_size=batch_size)
val_data = crear_mini_batches(X_val, y_val, batch_size=batch_size)
total_train_data = crear_mini_batches(X_train_total, y_train_total, batch_size=batch_size)
test_data = crear_mini_batches(X_test, y_test, batch_size=batch_size)
print(f"Creados {len(train_data)} mini-batches de tamaño {batch_size}")

# Transferir datos de prueba a GPU
X_test_gpu = cp.asarray(X_test)
y_test_gpu = cp.asarray(y_test)

# Definir la arquitectura de la red
print("Definiendo arquitectura de la red...")
net = [
    # Capa 1: Input 3x150x150 -> Conv 4x4 -> 32x147x147
    Convolution((3, 150, 150), 4, 32),      # 32 filtros de 4x4
    ReLU(),
    # Pooling: 32x147x147 -> 32x49x49
    Pooling(3, 3),                          # MaxPooling 3x3 (stride=3)

    # Capa 2: Conv 4x4 -> 64x46x46
    Convolution((32, 49, 49), 4, 64),
    ReLU(),
    # Pooling: 64x46x46 -> 64x15x15
    Pooling(3, 3),

    # Capa 3: Conv 4x4 -> 128x12x12
    Convolution((64, 15, 15), 4, 128),
    ReLU(),
    # Pooling: 128x12x12 -> 128x4x4
    Pooling(3, 3),

    # Capa 4: Conv 4x4 -> 128x1x1
    Convolution((128, 4, 4), 4, 128),
    ReLU(),

    # Flatten: 128x1x1 -> 128
    Reshape((128, 1, 1), 128),

    # Densas
    Dense(128, 512),
    ReLU(),
    Dense(512, 4),
    Softmax()
]

# Tiempo total de ejecución
end_time = time.time()
print(f"Tiempo total de ejecución: {end_time - start_time:.2f} segundos")

Iniciando carga de datos...
Cargando imágenes desde /content/dataset/Training...
  Clase glioma: 1321 imágenes cargadas
  Clase meningioma: 1339 imágenes cargadas
  Clase notumor: 1595 imágenes cargadas
  Clase pituitary: 1457 imágenes cargadas
Cargando imágenes desde /content/dataset/Testing...
  Clase glioma: 300 imágenes cargadas
  Clase meningioma: 306 imágenes cargadas
  Clase notumor: 405 imágenes cargadas
  Clase pituitary: 300 imágenes cargadas
X_train shape: (4569, 3, 150, 150)
y_train shape: (4569, 4)
X_val shape: (1143, 3, 150, 150)
y_val shape: (1143, 4)
X_test shape: (1311, 3, 150, 150)
y_test shape: (1311, 4)
Creando mini-batches y transfiriendo a GPU...
Creados 72 mini-batches de tamaño 64
Definiendo arquitectura de la red...
Tiempo total de ejecución: 36.52 segundos


## 4. Entrenamiento del modelo

In [11]:
# Entrenar la red
print("Iniciando entrenamiento...")
start_time = time.time()
epochs = 15
learning_rate = 0.1
history = train(train_data, val_data, net, categorical_cross_entropy, categorical_cross_entropy_prime, epochs, learning_rate)
ruta_modelo = 'modelo_cnn_cerebro.pkl'
guardar_modelo(net, ruta_modelo)

# Evaluar el modelo
print("Evaluando el modelo...")
y_true, y_pred = test(X_test_gpu, y_test_gpu, net)
acc = accuracy(y_true, y_pred)
print(f"Precisión del modelo: {acc * 100:.2f}%")

# Tiempo total de ejecución
end_time = time.time()
print(f"Tiempo total de ejecución: {end_time - start_time:.2f} segundos")

# Liberar memoria GPU
print("Liberando memoria GPU...")
cp.get_default_memory_pool().free_all_blocks()

Iniciando entrenamiento...
Epoch: 1/15 | Train Loss: 1.2225 | Train Acc: 0.4500 | Val Loss: 1.1403 | Val Acc: 0.4462
Tiempo de época: 252.36 segundos
Epoch: 2/15 | Train Loss: 0.8667 | Train Acc: 0.6459 | Val Loss: 0.9513 | Val Acc: 0.6124
Tiempo de época: 246.55 segundos
Epoch: 3/15 | Train Loss: 0.6913 | Train Acc: 0.7301 | Val Loss: 0.7237 | Val Acc: 0.6693
Tiempo de época: 247.31 segundos
Epoch: 4/15 | Train Loss: 0.6204 | Train Acc: 0.7676 | Val Loss: 0.7093 | Val Acc: 0.6824
Tiempo de época: 246.69 segundos
Epoch: 5/15 | Train Loss: 0.5674 | Train Acc: 0.7914 | Val Loss: 0.6756 | Val Acc: 0.7017
Tiempo de época: 248.15 segundos
Epoch: 6/15 | Train Loss: 0.5252 | Train Acc: 0.8109 | Val Loss: 0.6574 | Val Acc: 0.7253
Tiempo de época: 247.69 segundos
Epoch: 7/15 | Train Loss: 0.4810 | Train Acc: 0.8295 | Val Loss: 0.6638 | Val Acc: 0.7192
Tiempo de época: 248.04 segundos
Epoch: 8/15 | Train Loss: 0.4453 | Train Acc: 0.8420 | Val Loss: 0.9507 | Val Acc: 0.6264
Tiempo de época: 247.6

In [ ]:
net = [
    # Capa 1: Input 3x150x150 -> Conv 4x4 -> 32x147x147
    Convolution((3, 150, 150), 4, 32),      # 32 filtros de 4x4
    ReLU(),
    # Pooling: 32x147x147 -> 32x49x49
    Pooling(3, 3),                          # MaxPooling 3x3 (stride=3)

    # Capa 2: Conv 4x4 -> 64x46x46
    Convolution((32, 49, 49), 4, 64),
    ReLU(),
    # Pooling: 64x46x46 -> 64x15x15
    Pooling(3, 3),

    # Capa 3: Conv 4x4 -> 128x12x12
    Convolution((64, 15, 15), 4, 128),
    ReLU(),
    # Pooling: 128x12x12 -> 128x4x4
    Pooling(3, 3),

    # Capa 4: Conv 4x4 -> 128x1x1
    Convolution((128, 4, 4), 4, 128),
    ReLU(),

    # Flatten: 128x1x1 -> 128
    Reshape((128, 1, 1), 128),

    # Densas
    Dense(128, 512),
    ReLU(),
    Dense(512, 4),
    Softmax()
]

# Entrenar la red
print("Iniciando entrenamiento...")
start_time = time.time()
epochs = 40
learning_rate = 0.1
history = train(total_train_data, test_data, net, categorical_cross_entropy, categorical_cross_entropy_prime, epochs, learning_rate)
ruta_modelo = 'modelo_cnn_cerebro.pkl'
guardar_modelo(net, ruta_modelo)

# Evaluar el modelo
print("Evaluando el modelo...")
y_true, y_pred = test(X_test_gpu, y_test_gpu, net)
acc = accuracy(y_true, y_pred)
print(f"Precisión del modelo: {acc * 100:.2f}%")

# Tiempo total de ejecución
end_time = time.time()
print(f"Tiempo total de ejecución: {end_time - start_time:.2f} segundos")

# Liberar memoria GPU
print("Liberando memoria GPU...")
cp.get_default_memory_pool().free_all_blocks()

## 5. Evaluación de los resultados

In [ ]:
def analyze_model_performance(history, y_true_probs, y_pred_probs, class_names):
    """
    Análisis completo del rendimiento del modelo

    Args:
        history: Diccionario con el historial de entrenamiento
        y_true_probs: Array con las probabilidades verdaderas (one-hot encoded)
        y_pred_probs: Array con las probabilidades predichas
        class_names: Lista con los nombres de las clases

    Returns:
        dict: Diccionario con las métricas globales {
            'accuracy': float,
            'precision': float,
            'recall': float,
            'f1_score': float
        }
    """
    # Configuración para visualizaciones
    plt.style.use('seaborn-v0_8-whitegrid')
    plt.rcParams['font.family'] = 'DejaVu Sans'
    sns.set_palette("colorblind")

    # Convertir a numpy si son arrays de CuPy
    y_true_probs = cp.asnumpy(y_true_probs) if hasattr(y_true_probs, 'device') else y_true_probs
    y_pred_probs = cp.asnumpy(y_pred_probs) if hasattr(y_pred_probs, 'device') else y_pred_probs

    # Obtener clases predichas y verdaderas
    y_pred_class = np.argmax(y_pred_probs, axis=1)
    y_true_class = np.argmax(y_true_probs, axis=1)

    # 1. Visualización de curvas de entrenamiento (si hay historial)
    if history is not None:
        plot_training_history(history)

    # 2. Matriz de confusión
    plot_confusion_matrix(y_true_class, y_pred_class, class_names)

    # 3. Reporte de clasificación
    print_classification_report(y_true_class, y_pred_class, class_names)

    # 4. Visualización de curvas ROC y Precision-Recall
    plot_roc_curves(y_true_probs, y_pred_probs, class_names)
    plot_precision_recall_curves(y_true_probs, y_pred_probs, class_names)

    # 5. Análisis de distribución de probabilidades
    plot_probability_distributions(y_pred_probs, y_true_class, class_names)

    # 6. Análisis por clase
    analyze_class_performance(y_true_class, y_pred_class, y_pred_probs, class_names)

    # Calcular métricas globales
    report = classification_report(y_true_class, y_pred_class,
                                 target_names=class_names,
                                 output_dict=True)

    # Extraer métricas globales
    metrics = {
        'accuracy': report['accuracy'],
        'precision': report['macro avg']['precision'],
        'recall': report['macro avg']['recall'],
        'f1_score': report['macro avg']['f1-score']
    }

    # Mostrar resumen de métricas
    print("\n" + "="*50)
    print("Resumen de Métricas Globales".center(50))
    print("="*50)
    print(f"Accuracy:  {metrics['accuracy']:.4f}")
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall:    {metrics['recall']:.4f}")
    print(f"F1-Score:  {metrics['f1_score']:.4f}")
    print("="*50 + "\n")

    return metrics


def plot_training_history(history):
    """
    Visualiza las curvas de pérdida y precisión durante el entrenamiento
    """

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5), dpi=100)
    fig.patch.set_facecolor('#f8f9fa')

    # Colores profesionales
    train_color = '#2378f7'
    val_color = '#e84c3d'

    # Gráfica de pérdida
    ax1.plot(history['loss'], label='Entrenamiento', marker='o', markersize=6,
             linestyle='-', linewidth=2, color=train_color, alpha=0.9)
    ax1.plot(history['val_loss'], label='Validación', marker='s', markersize=6,
             linestyle='-', linewidth=2, color=val_color, alpha=0.9)
    ax1.set_title('Evolución de la Pérdida', fontsize=16, fontweight='bold', pad=15)
    ax1.set_xlabel('Época', fontsize=14, fontweight='bold')
    ax1.set_ylabel('Pérdida', fontsize=14, fontweight='bold')
    ax1.legend(fontsize=12, frameon=True, facecolor='white', edgecolor='#dddddd')
    ax1.grid(True, linestyle='--', alpha=0.7)
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)

    # Gráfica de precisión
    epochs = range(1, len(history['val_accuracy']) + 1)
    ax2.plot(epochs, history['val_accuracy'], label='Validación', marker='s', markersize=6,
             linestyle='-', linewidth=2, color=val_color, alpha=0.9)
    ax2.plot(epochs, history['accuracy'], label='Entrenamiento', marker='s', markersize=6,
             linestyle='-', linewidth=2, color=train_color, alpha=0.9)
    ax2.set_title('Evolución de la Precisión', fontsize=16, fontweight='bold', pad=15)
    ax2.set_xlabel('Época', fontsize=14, fontweight='bold')
    ax2.set_ylabel('Precisión', fontsize=14, fontweight='bold')
    ax2.legend(fontsize=12, frameon=True, facecolor='white', edgecolor='#dddddd')
    ax2.grid(True, linestyle='--', alpha=0.7)
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.show()

    # Análisis de convergencia
    best_epoch = np.argmax(history['val_accuracy']) + 1
    best_acc = max(history['val_accuracy'])

    print(f"Mejor rendimiento en validación: Época {best_epoch} con precisión {best_acc:.4f}")

    # Analizar la diferencia entre train y validation para detectar overfitting
    final_train_loss = history['loss'][-1]
    final_val_loss = history['val_loss'][-1]

    if final_train_loss < final_val_loss * 0.8:
        print("⚠️ Posible sobreajuste: La pérdida de entrenamiento es significativamente menor que la de validación.")
    elif final_train_loss > final_val_loss:
        print("⚠️ Comportamiento inusual: La pérdida de validación es menor que la de entrenamiento.")
    else:
        print("✓ No hay signos claros de sobreajuste basados en las curvas de pérdida.")


def plot_confusion_matrix(y_true, y_pred, class_names):
    """
    Visualiza la matriz de confusión
    """
    cm = confusion_matrix(y_true, y_pred)

    # Normalizar para obtener porcentajes
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    # Crear figura con fondo formal
    plt.figure(figsize=(10, 8), dpi=100)
    plt.rcParams['figure.facecolor'] = '#f8f9fa'

    # Crear mapa de calor con colores profesionales
    ax = sns.heatmap(cm_norm, annot=cm, fmt='d', cmap='Blues',
                     xticklabels=class_names, yticklabels=class_names,
                     cbar=False, annot_kws={"size": 14, "weight": "bold"})

    # Ajustar cuadrícula y bordes
    for _, spine in ax.spines.items():
        spine.set_visible(True)
        spine.set_linewidth(1.5)
        spine.set_edgecolor('#333333')

    # Ajustar etiquetas y título
    plt.title('Matriz de Confusión', fontsize=18, fontweight='bold', pad=20)
    plt.ylabel('Etiqueta Real', fontsize=16, fontweight='bold')
    plt.xlabel('Predicción', fontsize=16, fontweight='bold')

    # Ajustar tamaño de las etiquetas de ejes
    ax.tick_params(axis='both', which='major', labelsize=12)

    plt.tight_layout()
    plt.show()

    # Análisis de la matriz de confusión
    print("Análisis de la Matriz de Confusión:")

    # Diagonal principal (predicciones correctas)
    accuracy = np.trace(cm) / np.sum(cm)
    print(f"Precisión global: {accuracy:.4f} ({np.trace(cm)} de {np.sum(cm)} ejemplos)")

    # Análisis por clase
    for i, class_name in enumerate(class_names):
        precision = cm[i, i] / np.sum(cm[:, i]) if np.sum(cm[:, i]) > 0 else 0
        recall = cm[i, i] / np.sum(cm[i, :])
        print(f"Clase '{class_name}':")
        print(f"  - Precisión: {precision:.4f} ({cm[i, i]} de {np.sum(cm[:, i])} predicciones)")
        print(f"  - Recall: {recall:.4f} ({cm[i, i]} de {np.sum(cm[i, :])} ejemplos)")

        # Identificar confusiones más comunes
        if np.sum(cm[i, :]) - cm[i, i] > 0:  # Si hay al menos un error
            errors = [(j, cm[i, j]) for j in range(len(class_names)) if j != i and cm[i, j] > 0]
            errors.sort(key=lambda x: x[1], reverse=True)
            if errors:
                print(f"  - Confusiones más comunes: {class_name} confundido con", end=" ")
                for j, count in errors[:2]:  # Mostrar las 2 confusiones más comunes
                    print(f"{class_names[j]} ({count} veces, {count/np.sum(cm[i, :]):.1%})", end=", ")
                print()


def print_classification_report(y_true, y_pred, class_names):
    """
    Muestra el reporte de clasificación como una tabla visual
    """
    report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)

    # Extraer solo los datos de las clases (sin los promedios)
    class_data = {cls: report[cls] for cls in class_names}

    # Crear figura para la tabla
    plt.figure(figsize=(12, len(class_names) * 0.8 + 1.5), dpi=100)
    plt.rcParams['figure.facecolor'] = '#f8f9fa'

    # Crear tabla
    columns = ['Precisión', 'Recall', 'F1-Score', 'Accuracy', 'Muestras']
    cell_data = []

    for cls in class_names:
        # Para cada clase, extraer métricas
        precision = report[cls]['precision']
        recall = report[cls]['recall']
        f1 = report[cls]['f1-score']

        # Calcular accuracy específica para esta clase
        # (Nota: esto es una aproximación ya que sklearn no proporciona accuracy por clase)
        y_true_binary = (np.array(y_true) == list(class_names).index(cls))
        y_pred_binary = (np.array(y_pred) == list(class_names).index(cls))
        accuracy = np.mean(y_true_binary == y_pred_binary)

        # Obtener número de muestras
        support = report[cls]['support']

        cell_data.append([f"{precision:.4f}", f"{recall:.4f}", f"{f1:.4f}", f"{accuracy:.4f}", f"{support}"])

    # Crear y configurar la tabla
    table = plt.table(cellText=cell_data,
                     rowLabels=class_names,
                     colLabels=columns,
                     loc='center',
                     cellLoc='center',
                     bbox=[0.1, 0.1, 0.8, 0.8])

    # Formatear la tabla para un aspecto profesional
    table.auto_set_font_size(False)
    table.set_fontsize(12)
    table.scale(1, 1.5)  # Ajustar altura de filas

    # Estilo para encabezados de columnas
    for i in range(len(columns)):
        cell = table[0, i]
        cell.set_text_props(fontweight='bold', color='white')
        cell.set_facecolor('#2378f7')

    # Estilo para encabezados de filas
    for i in range(len(class_names)):
        cell = table[i+1, -1]
        cell.set_text_props(fontweight='bold')
        cell.set_facecolor('#f2f2f2')

    plt.title('Reporte de Clasificación', fontsize=18, fontweight='bold', pad=20)
    plt.axis('off')  # Ocultar ejes
    plt.tight_layout()
    plt.show()

    # Análisis del reporte
    print("\nAnálisis por Métrica:")
    print(f"Precisión media (macro): {report['macro avg']['precision']:.4f}")
    print(f"Recall medio (macro): {report['macro avg']['recall']:.4f}")
    print(f"F1-Score medio (macro): {report['macro avg']['f1-score']:.4f}")
    print(f"Accuracy: {report['accuracy']:.4f}")

    # Identificar clases con peor rendimiento
    class_f1 = {name: report[name]['f1-score'] for name in class_names}
    worst_class = min(class_f1.items(), key=lambda x: x[1])
    print(f"\nClase con peor rendimiento: '{worst_class[0]}' con F1-Score de {worst_class[1]:.4f}")


def plot_roc_curves(y_test, y_pred_probs, class_names):
    """
    Visualiza las curvas ROC para cada clase
    """
    n_classes = len(class_names)

    # Calcular ROC para cada clase
    fpr = {}
    tpr = {}
    roc_auc = {}

    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_test[:, i], y_pred_probs[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    # Crear figura con estilo profesional
    plt.figure(figsize=(12, 8), dpi=100)
    fig = plt.gcf()
    fig.patch.set_facecolor('#f8f9fa')
    ax = plt.gca()
    ax.set_facecolor('#f8f9fa')

    # Paleta de colores profesional
    colors = ['#2378f7', '#e84c3d', '#2ecc71', '#9b59b6', '#f39c12', '#1abc9c', '#34495e', '#e67e22']

    # Curva ROC para cada clase
    for i, cls in enumerate(class_names):
        color = colors[i % len(colors)]
        plt.plot(fpr[i], tpr[i], color=color, lw=2.5,
                 label=f'{cls} (AUC = {roc_auc[i]:.4f})')

    # Línea de referencia
    plt.plot([0, 1], [0, 1], 'k--', lw=1.5, alpha=0.7)

    # Configuración de ejes
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('Tasa de Falsos Positivos', fontsize=14, fontweight='bold')
    plt.ylabel('Tasa de Verdaderos Positivos', fontsize=14, fontweight='bold')
    plt.title('Curvas ROC por Clase', fontsize=18, fontweight='bold', pad=20)

    # Leyenda
    plt.legend(loc="lower right", fontsize=12, frameon=True,
               facecolor='white', edgecolor='#dddddd')

    # Cuadrícula y estilo
    plt.grid(True, linestyle='--', alpha=0.7)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(1.2)
    ax.spines['bottom'].set_linewidth(1.2)
    ax.tick_params(axis='both', which='major', labelsize=12)

    plt.tight_layout()
    plt.show()

    # Análisis de AUC
    print("\nAnálisis de AUC por Clase:")
    for i, cls in enumerate(class_names):
        print(f"Clase '{cls}': AUC = {roc_auc[i]:.4f}")

    # Promedio de AUC
    mean_auc = np.mean(list(roc_auc.values()))
    print(f"AUC promedio: {mean_auc:.4f}")


def plot_precision_recall_curves(y_test, y_pred_probs, class_names):
    """
    Visualiza las curvas de precisión-recall para cada clase
    """
    n_classes = len(class_names)

    # Calcular Precision-Recall para cada clase
    precision = {}
    recall = {}
    avg_precision = {}

    for i in range(n_classes):
        precision[i], recall[i], _ = precision_recall_curve(y_test[:, i], y_pred_probs[:, i])
        avg_precision[i] = average_precision_score(y_test[:, i], y_pred_probs[:, i])

    # Crear figura con estilo profesional
    plt.figure(figsize=(12, 8), dpi=100)
    fig = plt.gcf()
    fig.patch.set_facecolor('#f8f9fa')
    ax = plt.gca()
    ax.set_facecolor('#f8f9fa')

    # Paleta de colores profesional
    colors = ['#2378f7', '#e84c3d', '#2ecc71', '#9b59b6', '#f39c12', '#1abc9c', '#34495e', '#e67e22']

    # Curva Precision-Recall para cada clase
    for i, cls in enumerate(class_names):
        color = colors[i % len(colors)]
        plt.plot(recall[i], precision[i], color=color, lw=2.5,
                 label=f'{cls} (AP = {avg_precision[i]:.4f})')

    # Configuración de ejes
    plt.xlabel('Recall', fontsize=14, fontweight='bold')
    plt.ylabel('Precisión', fontsize=14, fontweight='bold')
    plt.title('Curvas de Precisión-Recall por Clase', fontsize=18, fontweight='bold', pad=20)

    # Leyenda
    plt.legend(loc="best", fontsize=12, frameon=True,
               facecolor='white', edgecolor='#dddddd')

    # Cuadrícula y estilo
    plt.grid(True, linestyle='--', alpha=0.7)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(1.2)
    ax.spines['bottom'].set_linewidth(1.2)
    ax.tick_params(axis='both', which='major', labelsize=12)

    plt.tight_layout()
    plt.show()

    # Se eliminó el Análisis de Precisión Media (AP) por Clase según lo solicitado


def plot_probability_distributions(y_pred_probs, y_true, class_names):
    """
    Analiza la distribución de probabilidades para predicciones correctas e incorrectas
    """
    # Obtener probabilidades máximas para cada predicción
    max_probs = np.max(y_pred_probs, axis=1)

    # Identificar predicciones correctas e incorrectas
    y_pred = np.argmax(y_pred_probs, axis=1)
    correct = (y_pred == y_true)

    # Crear figura con estilo profesional
    plt.figure(figsize=(10, 6), dpi=100)
    fig = plt.gcf()
    fig.patch.set_facecolor('#f8f9fa')
    ax = plt.gca()
    ax.set_facecolor('#f8f9fa')

    # Distribución de probabilidades con colores profesionales
    sns.histplot(max_probs[correct], bins=20, alpha=0.7, label='Correctas', color='#2ecc71')
    sns.histplot(max_probs[~correct], bins=20, alpha=0.7, label='Incorrectas', color='#e84c3d')

    # Configuración de ejes
    plt.xlabel('Probabilidad de la Clase Predicha', fontsize=14, fontweight='bold')
    plt.ylabel('Frecuencia', fontsize=14, fontweight='bold')
    plt.title('Distribución de Probabilidades: Correctas vs Incorrectas',
              fontsize=18, fontweight='bold', pad=20)

    # Leyenda
    plt.legend(fontsize=12, frameon=True, facecolor='white', edgecolor='#dddddd')

    # Cuadrícula y estilo
    plt.grid(True, linestyle='--', alpha=0.7)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(1.2)
    ax.spines['bottom'].set_linewidth(1.2)
    ax.tick_params(axis='both', which='major', labelsize=12)

    plt.tight_layout()
    plt.show()

    # Análisis estadístico de las probabilidades
    print("\nAnálisis de Probabilidades:")
    print(f"Probabilidad media para predicciones correctas: {np.mean(max_probs[correct]):.4f}")
    print(f"Probabilidad media para predicciones incorrectas: {np.mean(max_probs[~correct]):.4f}")

    # Umbral de confianza
    thresholds = [0.5, 0.7, 0.9]
    for threshold in thresholds:
        high_conf = max_probs >= threshold
        high_conf_correct = np.logical_and(high_conf, correct)

        if np.sum(high_conf) > 0:
            accuracy_high_conf = np.sum(high_conf_correct) / np.sum(high_conf)
            print(f"Precisión para predicciones con confianza ≥ {threshold}: {accuracy_high_conf:.4f} ({np.sum(high_conf_correct)} de {np.sum(high_conf)})")


def analyze_class_performance(y_true, y_pred, y_pred_probs, class_names):
    """
    Analiza el rendimiento del modelo para cada clase
    """
    # Crear un DataFrame con métricas por clase
    metrics_df = pd.DataFrame(index=class_names)

    # Calcular métricas por clase
    for i, cls in enumerate(class_names):
        # Predicciones binarias para esta clase
        y_true_cls = (y_true == i)
        y_pred_cls = (y_pred == i)

        # Cálculo de métricas
        TP = np.sum(np.logical_and(y_true_cls, y_pred_cls))
        FP = np.sum(np.logical_and(~y_true_cls, y_pred_cls))
        FN = np.sum(np.logical_and(y_true_cls, ~y_pred_cls))
        TN = np.sum(np.logical_and(~y_true_cls, ~y_pred_cls))

        # Calcular métricas
        accuracy = (TP + TN) / (TP + FP + FN + TN)
        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        recall = TP / (TP + FN) if (TP + FN) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

        # Probabilidades medias para esta clase
        cls_probs = y_pred_probs[:, i]
        mean_prob_correct = np.mean(cls_probs[y_true == i]) if np.any(y_true == i) else 0

        # Agregar métricas al DataFrame
        metrics_df.loc[cls, 'Precisión'] = precision
        metrics_df.loc[cls, 'Recall'] = recall
        metrics_df.loc[cls, 'F1-Score'] = f1
        metrics_df.loc[cls, 'Accuracy'] = accuracy
        metrics_df.loc[cls, 'Prob. Media (Correctas)'] = mean_prob_correct
        metrics_df.loc[cls, 'Ejemplos'] = np.sum(y_true == i)

    # Visualizar comparaciones entre clases con estilo profesional
    plt.figure(figsize=(14, 6), dpi=100)
    fig = plt.gcf()
    fig.patch.set_facecolor('#f8f9fa')
    ax = plt.gca()
    ax.set_facecolor('#f8f9fa')

    # Crear gráfica de barras con colores profesionales
    barplot = metrics_df[['Precisión', 'Recall', 'F1-Score']].plot(
        kind='bar',
        alpha=0.8,
        ax=ax,
        color=['#2378f7', '#2ecc71', '#9b59b6'],
        edgecolor='#333333',
        linewidth=1.2
    )

    # Configuración de ejes
    plt.title('Comparación de Métricas por Clase', fontsize=18, fontweight='bold', pad=20)
    plt.ylabel('Valor', fontsize=14, fontweight='bold')
    plt.xlabel('Clase', fontsize=14, fontweight='bold')

    # Leyenda
    plt.legend(fontsize=12, frameon=True, facecolor='white', edgecolor='#dddddd')

    # Cuadrícula y estilo
    plt.grid(True, linestyle='--', alpha=0.7, axis='y')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(1.2)
    ax.spines['bottom'].set_linewidth(1.2)
    ax.tick_params(axis='both', which='major', labelsize=12, rotation=0)

    # Añadir valores encima de las barras
    for container in barplot.containers:
        barplot.bar_label(container, fmt='%.2f', fontsize=10, fontweight='bold')

    plt.tight_layout()
    plt.show()

    # Identificar clases problemáticas
    worst_precision = metrics_df['Precisión'].idxmin()
    worst_recall = metrics_df['Recall'].idxmin()

    print("\nAnálisis por Clase:")
    print(f"Clase con menor precisión: '{worst_precision}' ({metrics_df.loc[worst_precision, 'Precisión']:.4f})")
    print(f"Clase con menor recall: '{worst_recall}' ({metrics_df.loc[worst_recall, 'Recall']:.4f})")

In [ ]:
metricas_baseline = analyze_model_performance(history, y_true, y_pred, clases)

## 6. Comparación con otros modelos

### 6.1. Mismo modelo en Tensorflow

In [ ]:
batch_size = 64
img_size = (150, 150)

train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    train_path,
    batch_size=batch_size,
    shuffle=True,
    image_size=img_size,
    label_mode='categorical'
)

test_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    test_path,
    batch_size=batch_size,
    image_size=img_size,
    label_mode='categorical'
)

normalization_layer = tf.keras.layers.Rescaling(1./255)

train_dataset = train_dataset.map(lambda x, y: (normalization_layer(x), y))
test_dataset = test_dataset.map(lambda x, y: (normalization_layer(x), y))

def to_float32(image, label):
    image = tf.cast(image, tf.float32)
    return image, label

train_dataset = train_dataset.map(to_float32)
test_dataset = test_dataset.map(to_float32)

# Esto precarga datos en memoria o disco para acelerar el entrenamiento
AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)

# Arquitectura del modelo (Idéntica al desarrollado)
tf_model = Sequential([
    # Capa 1: Input 3x150x150 -> Conv 4x4 -> 32x147x147
    layers.Conv2D(32, (4, 4), input_shape=(150, 150, 3)),  # filtros=32, kernel=4x4
    layers.ReLU(),
    layers.MaxPooling2D(pool_size=(3, 3), strides=3),      # 32x49x49

    # Capa 2: Conv 4x4 -> 64x46x46
    layers.Conv2D(64, (4, 4)),
    layers.ReLU(),
    layers.MaxPooling2D(pool_size=(3, 3), strides=3),      # 64x15x15

    # Capa 3: Conv 4x4 -> 128x12x12
    layers.Conv2D(128, (4, 4)),
    layers.ReLU(),
    layers.MaxPooling2D(pool_size=(3, 3), strides=3),      # 128x4x4

    # Capa 4: Conv 4x4 -> 128x1x1
    layers.Conv2D(128, (4, 4)),
    layers.ReLU(),

    # Flatten
    layers.Flatten(),  # De 128x1x1 a 128

    # Capas densas
    layers.Dense(512),
    layers.ReLU(),
    layers.Dense(4),
    layers.Softmax()
])

# Mostrar resumen del modelo
tf_model.summary()

In [ ]:
optimizer = SGD(learning_rate=0.1) # Sin optimizador y mismo learning rate

tf_model.compile(
    optimizer=optimizer,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = tf_model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=40, # Mismo número de epochs
    verbose=1
)

In [ ]:
# Evaluar el modelo
y_true_probs = []
y_pred_probs = []

for images, labels in test_dataset:
    y_true_probs.append(labels.numpy())
    y_pred_probs.append(tf_model.predict(images, verbose=0))

y_true_probs = np.concatenate(y_true_probs)
y_pred_probs = np.concatenate(y_pred_probs)

metricas_tf_model = analyze_model_performance(history.history, y_true_probs, y_pred_probs, clases)

### 6.1. Modelo preentrenado (Resnet-50)

In [ ]:
# Cargar el modelo preentrenado (sin las capas de clasificación)
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(150, 150, 3))

# Añadir capas personalizadas para nuestro problema
resnet_model = Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),  # Reducción de dimensiones
    layers.Dense(512, activation='relu'),  # Capa densa
    layers.Dense(4, activation='softmax')  # 4 clases para clasificación (ajusta esto)
])

# Ver el resumen del modelo
resnet_model.summary()

In [ ]:
resnet_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = resnet_model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=20
)

In [ ]:
# Evaluar el modelo
y_true_probs = []
y_pred_probs = []

for images, labels in test_dataset:
    y_true_probs.append(labels.numpy())
    y_pred_probs.append(resnet_model.predict(images, verbose=0))

y_true_probs = np.concatenate(y_true_probs)
y_pred_probs = np.concatenate(y_pred_probs)

metricas_resnet = analyze_model_performance(history.history, y_true_probs, y_pred_probs, clases)

## 7. Comparación de los 3 modelos

In [ ]:
def plot_model_comparison(metrics_dict1, metrics_dict2, metrics_dict3, model_names):
    """
    Compara visualmente las métricas globales de tres modelos diferentes.

    Args:
        metrics_dict1: Diccionario de métricas del primer modelo
        metrics_dict2: Diccionario de métricas del segundo modelo
        metrics_dict3: Diccionario de métricas del tercer modelo
        model_names: Lista con los nombres de los modelos [nombre1, nombre2, nombre3]
    """
    # Configuración del estilo
    plt.style.use('seaborn-v0_8-whitegrid')
    plt.rcParams['font.family'] = 'DejaVu Sans'
    plt.rcParams['axes.titlesize'] = 14
    plt.rcParams['axes.labelsize'] = 12

    # Preparar datos
    metrics = ['accuracy', 'precision', 'recall', 'f1_score']
    metric_labels = ['Accuracy', 'Precision', 'Recall', 'F1-Score']

    # Valores para cada modelo
    model1_values = [metrics_dict1[m] for m in metrics]
    model2_values = [metrics_dict2[m] for m in metrics]
    model3_values = [metrics_dict3[m] for m in metrics]

    # Configuración del gráfico
    x = np.arange(len(metric_labels))  # ubicaciones de las etiquetas
    width = 0.25  # ancho de las barras

    fig, ax = plt.subplots(figsize=(12, 6))

    # Crear barras para cada modelo
    rects1 = ax.bar(x - width, model1_values, width, label=model_names[0], color='#1f77b4')
    rects2 = ax.bar(x, model2_values, width, label=model_names[1], color='#ff7f0e')
    rects3 = ax.bar(x + width, model3_values, width, label=model_names[2], color='#2ca02c')

    # Añadir etiquetas, título y leyenda
    ax.set_ylabel('Score', fontsize=12)
    ax.set_title('Comparación de Métricas entre Modelos', fontweight='bold', pad=20)
    ax.set_xticks(x)
    ax.set_xticklabels(metric_labels)
    ax.legend(frameon=True, facecolor='white')
    ax.set_ylim(0, 1.1)

    # Añadir valores en las barras
    def autolabel(rects):
        for rect in rects:
            height = rect.get_height()
            ax.annotate(f'{height:.3f}',
                        xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 3),  # 3 points vertical offset
                        textcoords="offset points",
                        ha='center', va='bottom',
                        fontsize=10)

    autolabel(rects1)
    autolabel(rects2)
    autolabel(rects3)

    # Ajustes finales
    plt.tight_layout()
    plt.show()

model_names = ['CNN TFG', 'CNN Tensorflow', 'ResNet']
plot_model_comparison(metricas_baseline, metricas_tf_model, metricas_resnet, model_names)